# Wallclock Distribution Analysis

Statistical distribution analysis of `wallclock_requested` (a.k.a. `requested_seconds`) across
all runtime-prediction datasets. This analysis characterizes the natural clustering of wallclock
values at HPC partition limits, and assesses whether these clusters define meaningful job
populations with distinct runtime characteristics.

**Purpose:** Inform whether a mixture-of-experts approach (separate models per wallclock cluster)
would improve runtime prediction beyond what a single model with wallclock as a feature already achieves.

**Prerequisites:** This notebook requires downloaded datasets. If not available, run from `workspace/`:
```bash
hpc-oda datasets prepare <descriptor.yml> --cache .hpc_oda/cache/datasets --out .
```

**Related:** Issue [#124](https://github.com/NatLabRockies/hpc-oda-commons/issues/124)

## 1. Setup

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# Paths
REPO_ROOT = Path.cwd().parent.parent  # from docs/benchmarking/
WORKSPACE = REPO_ROOT / 'workspace'
DATA_DIR  = WORKSPACE / 'data' / 'datasets'
CARDS_DIR = REPO_ROOT / 'docs' / 'benchmarking' / 'datasets'

# Datasets with wallclock field and their benchmark windows
DATASETS = {
    'nlr_kestrel':     ('2025-03-29', '2025-06-26'),
    'nlr_eagle':       ('2023-02-01', '2023-05-01'),
    'lassen':          ('2020-03-17', '2020-06-14'),
    'fdata_fugaku':    ('2023-06-30', '2023-09-27'),
    'pm100':           ('2020-06-13', '2020-09-10'),
    'ccin2p3_2024':    ('2024-09-09', '2024-12-07'),
    'atlas_mustang':   ('2015-08-08', '2015-11-05'),
    'atlas_opentrinity': ('2016-02-03', '2016-04-22'),
}

# Wallclock column name varies by dataset
WALLCLOCK_FIELDS = ['requested_seconds', 'wallclock_requested', 'wallclock_requested_seconds']

In [ ]:
def slice_to_window(table, window_start, window_end):
    """Slice a table to the benchmark window using overlap predicate."""
    cols = table.column_names
    submit_col = 'submit_time' if 'submit_time' in cols else 'start_time'
    end_col = 'end_time' if 'end_time' in cols else submit_col
    lo = datetime.strptime(window_start, '%Y-%m-%d').replace(tzinfo=timezone.utc)
    hi = datetime.strptime(window_end, '%Y-%m-%d').replace(tzinfo=timezone.utc) + timedelta(days=1)
    sc = table.column(submit_col)
    ec = table.column(end_col)
    mask = pc.and_(
        pc.less(sc, pa.scalar(hi, type=sc.type)),
        pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
    )
    return table.filter(mask, null_selection_behavior='drop')


def find_wallclock_col(df):
    """Find the wallclock column name in a dataframe."""
    for col in WALLCLOCK_FIELDS:
        if col in df.columns:
            return col
    return None


# Load all datasets
loaded = {}
for name, (ws, we) in DATASETS.items():
    path = DATA_DIR / name / 'data.parquet'
    if not path.exists():
        print(f'  {name}: not found, skipping')
        continue
    table = pq.read_table(path)
    sliced = slice_to_window(table, ws, we)
    df = sliced.to_pandas()
    wc_col = find_wallclock_col(df)
    if wc_col is None:
        print(f'  {name}: no wallclock column found, skipping')
        continue
    # Filter to valid wallclock values
    df = df[df[wc_col].notna() & (df[wc_col] > 0)].copy()
    df['wc_hours'] = df[wc_col] / 3600
    loaded[name] = {'df': df, 'wc_col': wc_col}
    print(f'  {name}: {len(df):,} rows loaded (window {ws} to {we}), wallclock col = {wc_col}')

print(f'\nLoaded {len(loaded)} / {len(DATASETS)} datasets')

## 2. Per-Dataset Wallclock Histograms

These histograms show the distribution of requested wallclock values on a log-scale x-axis.
The spikes at round-number hours represent HPC partition limits — users submit to a partition
with a maximum wallclock, and request at or near that maximum as a safety buffer.

In [ ]:
n_datasets = len(loaded)
n_cols = 2
n_rows_grid = (n_datasets + 1) // 2

fig, axes = plt.subplots(n_rows_grid, n_cols, figsize=(16, 4 * n_rows_grid))
axes = axes.flat

for idx, (name, data) in enumerate(loaded.items()):
    ax = axes[idx]
    wc_h = data['df']['wc_hours']
    
    # Log-scale histogram
    log_bins = np.logspace(np.log10(max(wc_h.min(), 0.01)), np.log10(wc_h.max()), 100)
    ax.hist(wc_h, bins=log_bins, color='steelblue', edgecolor='none')
    ax.set_xscale('log')
    ax.set_xlabel('Wallclock requested (hours, log scale)')
    ax.set_ylabel('Jobs')
    ax.set_title(f'{name} ({len(wc_h):,} jobs)')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K' if x >= 1000 else f'{x:.0f}'))
    
    # Annotate top spikes
    # Find values where > 5% of jobs cluster (within ±2 min = ±0.033h)
    wc_min = (data['df'][data['wc_col']] / 60).round().astype(int)
    from collections import Counter
    counts = Counter(wc_min.values)
    total = len(wc_min)
    for mins, count in counts.most_common(5):
        pct = count / total * 100
        if pct >= 3:
            hours = mins / 60
            ax.axvline(hours, color='red', linestyle='--', alpha=0.7, linewidth=0.8)
            ax.text(hours * 1.05, ax.get_ylim()[1] * 0.85, f'{hours:.0f}h\n{pct:.0f}%',
                    fontsize=8, color='red')

# Hide unused subplots
for idx in range(n_datasets, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Wallclock Requested Distributions — All Datasets', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Heatmap: Requested Wallclock vs Actual Runtime

Each plot shows the density of jobs at each (requested_wallclock, actual_runtime) coordinate.
Both axes use log scale. The diagonal dashed line represents "used 100% of requested time."
Jobs below the diagonal finished early; jobs above it exceeded their request (rare — usually
the scheduler kills them at the limit).

Dense clusters at specific x-values (vertical bands) confirm the partition limit spikes.
The vertical spread within each band shows how variable actual runtime is for that wallclock group.

In [ ]:
n_datasets = len(loaded)
n_cols = 2
n_rows_grid = (n_datasets + 1) // 2

fig, axes = plt.subplots(n_rows_grid, n_cols, figsize=(16, 5 * n_rows_grid))
axes = axes.flat

for idx, (name, data) in enumerate(loaded.items()):
    ax = axes[idx]
    df = data['df']
    wc_col = data['wc_col']
    
    wc = df[wc_col].values
    rt = df['runtime_seconds'].values
    # Filter to positive values for log scale
    valid = (wc > 0) & (rt > 0) & np.isfinite(wc) & np.isfinite(rt)
    wc_v, rt_v = wc[valid], rt[valid]
    
    # 2D histogram in log space
    h = ax.hist2d(
        np.log10(wc_v / 3600),  # x: log10(hours)
        np.log10(rt_v / 3600),  # y: log10(hours)
        bins=80,
        cmap='YlOrRd',
        cmin=1,
    )
    
    # Diagonal: used 100% of requested
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='used 100%')
    
    ax.set_xlabel('Requested wallclock (log10 hours)')
    ax.set_ylabel('Actual runtime (log10 hours)')
    ax.set_title(f'{name} ({len(wc_v):,} jobs)')
    
    # Readable tick labels
    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_major_formatter(ticker.FuncFormatter(
            lambda x, _: f'{10**x:.1f}h' if x <= 2 else f'{10**x:.0f}h'))
    ax.legend(fontsize=8, loc='upper left')

for idx in range(n_datasets, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Requested Wallclock vs Actual Runtime — 2D Density (log-log)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Spike Detection

Identify exact wallclock values where a significant fraction (>1%) of jobs cluster.
These spikes correspond to partition time limits — the system's configured maximum
wallclock for each queue. The tolerance is ±2 minutes to catch slight rounding variations.

In [ ]:
from collections import Counter

all_spikes = {}  # name -> list of (hours, count, pct)

for name, data in loaded.items():
    df = data['df']
    wc_col = data['wc_col']
    
    # Round to nearest minute
    wc_min = (df[wc_col] / 60).round().astype(int)
    total = len(wc_min)
    counts = Counter(wc_min.values)
    
    spikes = []
    for mins, count in counts.most_common(20):
        pct = count / total * 100
        if pct >= 1.0:  # threshold: >1% of jobs
            spikes.append((mins / 60, count, pct))
    
    all_spikes[name] = spikes
    
    print(f'\n=== {name} ({total:,} jobs) ===')
    print(f'{"Hours":>8} {"Count":>10} {"Percentage":>10}')
    print(f'{"-"*8} {"-"*10} {"-"*10}')
    for hours, count, pct in sorted(spikes, key=lambda x: -x[2]):
        print(f'{hours:>8.1f} {count:>10,} {pct:>9.1f}%')